# Boca Raton Data Cleaning Pipeline
## City-Specific Data Normalization

This notebook handles **Boca Raton specific** data cleaning and normalization:
- Loads raw CSV files from `results_folder/bocaraton/tables/`
- Removes header repetitions and description-only rows
- Standardizes column names and data types
- Applies Boca Raton-specific field mappings
- Outputs clean, normalized data ready for consolidation

**Input**: Raw CSV files from extraction pipeline  
**Output**: Clean, normalized CSV ready for joining with other cities

## Environment Setup

In [ ]:
import pandas as pd
from pathlib import Path
import sys

# Project paths
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[1]
else:
    cwd = Path.cwd()
    ROOT = cwd.parent if cwd.name == "src" else cwd

RESULTS_DIR = ROOT / "results_folder"
BOCA_DIR = RESULTS_DIR / "bocaraton"
CLEAN_DIR = ROOT / "clean_data"
CLEAN_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("BOCA_DIR:", BOCA_DIR)
print("CLEAN_DIR:", CLEAN_DIR)

## Load Raw Boca Raton Data
Load all CSV files from the tables directory and combine them.

In [ ]:
# Load all Boca Raton CSV files
tables_dir = BOCA_DIR / "tables"
csv_files = list(tables_dir.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files in {tables_dir}")
for f in csv_files:
    print(f" - {f.name}")

# Combine all CSV files
dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    df["source_file"] = csv_file.name
    dfs.append(df)
    print(f"Loaded {csv_file.name}: {len(df)} rows, {len(df.columns)} columns")

if dfs:
    boca_raw = pd.concat(dfs, ignore_index=True, sort=False)
    print(f"\n✅ Combined dataset: {len(boca_raw)} rows, {len(boca_raw.columns)} columns")
    print("Columns:", list(boca_raw.columns))
else:
    print("❌ No CSV files found!")

## Data Cleaning - Remove Headers and Invalid Rows
Clean up Boca Raton specific data issues like repeated headers and description-only rows.

In [ ]:
boca_clean = boca_raw.copy()

print("🔍 Before cleaning:")
print(f"Rows: {len(boca_clean)}")
print(f"Null rates (top 5):\n{boca_clean.isna().mean().sort_values(ascending=False).head()}")

# Remove obvious header repeats (rows whose first col literally equals the header name)
if "Main Address" in boca_clean.columns:
    mask_headers = boca_clean["Main Address"].astype(str).str.strip().eq("Main Address")
    rows_before = len(boca_clean)
    boca_clean = boca_clean[~mask_headers]
    print(f"Removed {rows_before - len(boca_clean)} header repeat rows")

# Drop the description-only rows that slipped under "Main Address"
if "Main Address" in boca_clean.columns:
    mask_descr = boca_clean["Main Address"].astype(str).str.startswith("Description:", na=False)
    rows_before = len(boca_clean)
    boca_clean = boca_clean[~mask_descr]
    print(f"Removed {rows_before - len(boca_clean)} description-only rows")

# Convert date columns
date_columns = ["Resolved Date", "Opened Date", "Closed Date", "Compliance Date", "Citation Issued"]
for col in date_columns:
    if col in boca_clean.columns:
        boca_clean[col] = pd.to_datetime(boca_clean[col], errors="coerce")
        print(f"Converted {col} to datetime")

print(f"\n✅ After cleaning: {len(boca_clean)} rows remaining")
print(f"Sample data:\n{boca_clean.head(3)}")

## Normalize to Standard Schema
Map Boca Raton columns to the standardized schema used across all cities.

In [ ]:
# Standard schema for all cities
normalized_cols = [
    "case_id_raw", "address_raw", "parcel_raw",
    "violation", "violation_status", "citation_issued",
    "compliance_date", "resolved_date", "opened_date", "closed_date",
    "assigned_to", "project", "district", "violation_fee_total",
    "city", "source_file"
]

# Create normalized DataFrame
boca_normalized = pd.DataFrame(index=boca_clean.index, columns=normalized_cols)

# Map Boca Raton specific columns to normalized schema
column_mapping = {
    "Main Address": "address_raw",
    "Resolved Date": "resolved_date", 
    "Opened Date": "opened_date",
    "Closed Date": "closed_date",
    "Compliance Date": "compliance_date",
    "Citation Issued": "citation_issued",
    "Violation": "violation",
    "Violation Status": "violation_status",
    "Parcel": "parcel_raw",
    "Project": "project",
    "Assigned To": "assigned_to",
    "District": "district"
}

# Apply mapping
for boca_col, norm_col in column_mapping.items():
    if boca_col in boca_clean.columns:
        boca_normalized[norm_col] = boca_clean[boca_col]
        print(f"Mapped: {boca_col} -> {norm_col}")

# Add metadata
boca_normalized["city"] = "Boca Raton"
boca_normalized["source_file"] = boca_clean["source_file"]

print(f"\n✅ Normalized schema applied")
print(f"Non-null values per column:")
for col in normalized_cols:
    non_null = boca_normalized[col].notna().sum()
    if non_null > 0:
        print(f"  {col}: {non_null}")

print(f"\nSample normalized data:")
print(boca_normalized[["address_raw", "violation", "resolved_date", "city"]].head(3))

## Save Clean Data
Save the cleaned and normalized Boca Raton data for consolidation.

In [ ]:
# Save cleaned Boca Raton data
output_file = CLEAN_DIR / "boca_raton_clean.csv"
boca_normalized.to_csv(output_file, index=False)

print(f"✅ Saved clean Boca Raton data to: {output_file}")
print(f"📊 Final stats:")
print(f"  - Rows: {len(boca_normalized)}")
print(f"  - Columns: {len(boca_normalized.columns)}")
print(f"  - Date range: {boca_normalized['resolved_date'].min()} to {boca_normalized['resolved_date'].max()}")
print(f"  - Unique addresses: {boca_normalized['address_raw'].nunique()}")

# Quick validation
print(f"\n🔍 Data validation:")
print(f"  - Missing addresses: {boca_normalized['address_raw'].isna().sum()}")
print(f"  - Missing violations: {boca_normalized['violation'].isna().sum()}")
print(f"  - All rows have city='Boca Raton': {(boca_normalized['city'] == 'Boca Raton').all()}")

print(f"\n🎯 Ready for consolidation! Next: run other city cleaning notebooks, then 3_consolidation.ipynb")